# IndustReal: data and annotations


## Section 1 - Purpose

This notebook prepares the public dataset IndustReal (Schoonbeek et al. 2024) for the pipeline of this thesis. IndustReal is the first public test of the Phase 1 pipeline: frozen C-RADIOv4 with prototypes from a small support set. If perception holds, the tracking of Phase 2 is tried on it as it stands.

IndustReal contains 84 egocentric recordings of 27 participants, made with a HoloLens 2, who assemble and maintain a toy car built from 3D-printed parts. The data and the code are released under the Apache 2.0 license. Only the RGB images and the annotations are used here.

The recordings are split by participant into train, validation and test, as in the paper. The support set will come from train, the validation frames from validation, and the test frames from test, so that reference frames never come from recordings used for evaluation.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

Schoonbeek et al. 2024, IndustReal (WACV), Section 4.3: 27 participants; Section 4.4: the split is made on participants, 12 for training, 5 for validation and 10 for testing, to ensure variation in viewpoint, execution and head movements between the three sets.

IndustReal code repository (github.com/TimSchoonbeek/IndustReal), readme: 84 recordings; readme and license.txt: data, code and weights under the Apache 2.0 license.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 2 - Configuration

In [22]:
# 2.1 Configuration
import json
from pathlib import Path

from remotezip import RemoteZip

# Project folder: the first folder above this notebook that contains src/
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())

# IndustReal on 4TU.ResearchData (doi 10.4121/b008dd74-020d-4ea4-a8ba-7bb60769d224, version 2, Apache-2.0)
DATASET_URL = "https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/"
ZIP_IDS = {                                    # one zip per part of each split, as listed on the dataset page
    "train_p1": "f58a5f18-9a71-4443-8a48-e59530df9ee8",
    "train_p2": "5aae5141-cece-41db-9681-47acd5300705",
    "train_p3": "4e2ed830-a1fe-40fa-9c5a-2aeae3964616",
    "train_p4": "3ff5409e-165f-4354-9fa4-a4fde639a7a6",
    "val_p1": "bb336949-248c-4ae6-82ef-107dbe61d10f",
    "val_p2": "e899de9f-313a-482b-a6b5-e2de4f38412e",
    "test_p1": "ec7a4f94-7c1d-497d-b75a-4e694b7a9f7e",
    "test_p2": "d07789b6-83b1-429d-be0c-95edba210757",
    "test_p3": "31f88cb6-a2e7-44df-ac51-5aaa0a148fc3",
}
ANNOTATION_PREFIXES = ("AR_labels", "OD_labels", "PSR_labels")   # only the annotations; sensor data (gaze, hands, pose) when needed

# List of the 33 procedure steps and the 11 components, from the IndustReal code repository
PROCEDURE_INFO_URL = "https://raw.githubusercontent.com/TimSchoonbeek/IndustReal/ad86a35d4b5125739e24af677e5d7b55c74af945/PSR/procedure_info.json"   # pinned to the last commit (19 Aug 2024)

DATA_DIR = PROJECT_ROOT / "data" / "industreal"       # not tracked by git (data/* in .gitignore)
RECORDINGS_DIR = DATA_DIR / "recordings"              # one folder per split, one subfolder per recording
PROCEDURE_INFO_PATH = DATA_DIR / "procedure_info.json"
MANIFEST_PATH = DATA_DIR / "annotation_manifest.json"   # list of the annotation files downloaded, per zip
VIDEOS_ZIP_ID = "cf21602e-4424-42a4-b05d-9897957082ad"          # all_rgb_videos.zip on the dataset page
VIDEOS_ZIP_MD5 = "c5b8901dba179d2eb10e9348f8163c79"             # MD5 given on the dataset page
VIDEOS_ZIP_PATH = DATA_DIR / "all_rgb_videos.zip"
VIDEOS_DIR = DATA_DIR / "videos"                                # one mp4 per recording, 10 fps
RGB_COUNTS_PATH = DATA_DIR / "rgb_frame_counts.json"            # number of RGB jpg frames per recording, from the zip indexes
FPS = 10                                                        # RGB frame rate (paper, Section 4.2; checked on all videos in 5.5)

print("Project folder:", PROJECT_ROOT)
print("Data folder exists:", DATA_DIR.is_dir())

Project folder: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project
Data folder exists: True


------------------------------------------------------------------------------------------------------------------------------------------

## Section 3 - Annotations

The dataset is published as nine zip files of recordings, three to four per split, of 3.8 to 10.2 GB each. Each recording folder holds every sensor of the headset (RGB, stereo, depth, ambient light), the tracking of gaze, hands and head, and the annotations.

Only the annotation files are needed now. A zip file keeps its index at the end, so the index can be read on the server and single files can be extracted without downloading the whole archive. The sensor data and the RGB frames are downloaded later, and only for the recordings that are used.

The list of the 33 procedure steps and of the 11 components comes from the code repository of the dataset (PSR/procedure_info.json), at its last commit.

In [2]:
# 3.1 Download of the annotations (only the files that are missing)
import urllib.request

def download_annotations():
    """Extract the annotation files of every recording from the remote zips and write the manifest."""
    manifest = {}
    for part, file_id in ZIP_IDS.items():
        split = part.split("_")[0]                                     # train, val or test
        with RemoteZip(DATASET_URL + file_id) as z:                   # reads the zip index on the server
            names = [n for n in z.namelist() if n.split("/")[-1].startswith(ANNOTATION_PREFIXES)]
            for n in names:
                if not (RECORDINGS_DIR / split / n).exists():
                    z.extract(n, RECORDINGS_DIR / split)
        manifest[part] = [f"{split}/{n}" for n in names]
        print(part, len(names), "annotation files")
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=1))

manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
if manifest is None or not all((RECORDINGS_DIR / f).exists() for files in manifest.values() for f in files):
    download_annotations()
    manifest = json.loads(MANIFEST_PATH.read_text())
else:
    print("All annotation files already present, no download")
print("Annotation files:", sum(len(files) for files in manifest.values()))

if not PROCEDURE_INFO_PATH.exists():
    urllib.request.urlretrieve(PROCEDURE_INFO_URL, PROCEDURE_INFO_PATH)
print("Procedure info present:", PROCEDURE_INFO_PATH.exists())

All annotation files already present, no download
Annotation files: 420
Procedure info present: True


Observation 3.1: The nine zip files hold 420 annotation files, five for each of the 84 recordings: AR_labels.csv, OD_labels.json, PSR_labels.csv, PSR_labels_with_errors.csv and PSR_labels_raw.csv. The recordings are 36 in train, 16 in validation and 32 in test, with 12, 5 and 10 participants, as in the paper.

Inside the zips the recording folders sit at the top level, without the recordings/train, val and test folders shown in the readme, so the split is taken from the name of the zip. Some image folders also contain Thumbs.db, a Windows system file, which is not part of the data.

The names of the recordings have the form participant_procedure_instruction_take, for example 05_assy_0_1 or 14_main_2_3: assy is the assembly and main the maintenance. Every participant has an assy_0 recording and all but participant 15 a main_0 recording. Twenty-two participants have one more recording, four have two (10, 13, 14 and 27), participant 15 has two maintenance recordings with instruction 3, and participant 21 has none  (both missing recordings exist as videos only, see Section 5). 

------------------------------------------------------------------------------------------------------------------------------------------

References Section 3:

IndustReal code repository (github.com/TimSchoonbeek/IndustReal), readme, Data information and Annotation information: the content of each recording folder and the meaning of the five annotation files; PSR/procedure_info.json, at commit ad86a35 (19 August 2024): the 33 procedure steps and the 11 components.

Schoonbeek et al. 2024, IndustReal (WACV), Section 4.3: each participant performed a correct assembly instruction, a correct maintenance instruction and one or two instructions with errors.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 4 - Components and procedure steps

IndustReal does not annotate single parts. It divides the toy car into 11 components, each made of one or more parts, and describes the state of the assembly as the state of each component: installed correctly, not installed yet, or installed incorrectly. The procedure steps are defined on the same components: each component can be installed, installed incorrectly or removed, which gives the 33 steps of the procedure.

The components are what Phase 2 and Phase 3 will reason about, so their list and their order are read first. Whether the perception of Phase 1 can work on whole components or needs the single parts is decided later, on the frames.

In [3]:
# 4.1 The 11 components and their procedure steps
procedure = json.loads(PROCEDURE_INFO_PATH.read_text())
print(len(procedure), "procedure steps")

COMPONENTS = []                                                   # component names, in the order of the state vector
for idx in sorted({s["state_idx"] for s in procedure}):
    steps = [s for s in procedure if s["state_idx"] == idx]
    install = next(s for s in steps if s["install"] and not s["description"].startswith("Incorrectly"))
    COMPONENTS.append(install["description"].removeprefix("Install "))
    print(idx, COMPONENTS[-1], "| step ids", [s["id"] for s in steps],
          "| expected in assembly:", install["expected_in_assy"], "| in maintenance:", install["expected_in_main"])

33 procedure steps
0 base | step ids [0, 1, 2] | expected in assembly: False | in maintenance: False
1 front chassis | step ids [3, 4, 5] | expected in assembly: True | in maintenance: False
2 front chassis pin | step ids [6, 7, 8] | expected in assembly: True | in maintenance: False
3 rear chassis | step ids [9, 10, 11] | expected in assembly: True | in maintenance: False
4 short rear chassis | step ids [12, 13, 14] | expected in assembly: False | in maintenance: True
5 front rear chassis pin | step ids [15, 16, 17] | expected in assembly: True | in maintenance: True
6 rear rear chassis pin | step ids [18, 19, 20] | expected in assembly: True | in maintenance: True
7 front bracket | step ids [21, 22, 23] | expected in assembly: True | in maintenance: False
8 front bracket screw | step ids [24, 25, 26] | expected in assembly: True | in maintenance: False
9 front wheel assy | step ids [27, 28, 29] | expected in assembly: True | in maintenance: False
10 rear wheel assy | step ids [30, 31

Observation 4.1: The procedure has 33 steps on 11 components, three per component with consecutive ids: installed (3 × index), installed incorrectly (3 × index + 1) and removed (3 × index + 2). The base is expected in neither procedure, since it is on the desk from the start. The short rear chassis is expected only in maintenance, the front chassis, its pin, the front bracket, its screw and the front wheel only in assembly, and the two pins of the rear chassis and the rear wheel in both. This agrees with the paper, where maintenance replaces the long braces of the rear chassis with short ones.

In [4]:
# 4.2 Erroneous steps per recording, by instruction
import csv

def read_rows(path):
    """Rows of a csv annotation file, as tuples, without empty lines."""
    return [tuple(r) for r in csv.reader(open(path)) if r]

recordings = sorted(p for split in ("train", "val", "test") for p in (RECORDINGS_DIR / split).iterdir() if p.is_dir())
summary = []
for rec in recordings:
    participant, task, instruction, take = rec.name.split("_")
    correct = read_rows(rec / "PSR_labels.csv")
    with_errors = read_rows(rec / "PSR_labels_with_errors.csv")
    errors = [r for r in with_errors if r not in correct]           # steps that appear only in the file with errors
    summary.append((rec.parent.name, rec.name, task, instruction, len(correct), len(errors)))

for label, rows in [("instruction 0", [s for s in summary if s[3] == "0"]), ("other instructions", [s for s in summary if s[3] != "0"])]:
    print(label, ":", len(rows), "recordings,", sum(s[5] > 0 for s in rows), "with erroneous steps")
print()
for s in summary:
    if s[5] > 0 or s[3] != "0":
        print(s)

instruction 0 : 53 recordings, 5 with erroneous steps
other instructions : 31 recordings, 25 with erroneous steps

('test', '03_assy_1_3', 'assy', '1', 10, 2)
('test', '08_assy_0_1', 'assy', '0', 10, 2)
('test', '08_assy_2_4', 'assy', '2', 8, 1)
('test', '08_main_0_1', 'main', '0', 10, 1)
('test', '09_assy_3_1', 'assy', '3', 8, 1)
('test', '10_assy_3_2', 'assy', '3', 15, 1)
('test', '10_main_3_4', 'main', '3', 8, 0)
('test', '12_assy_3_4', 'assy', '3', 8, 1)
('test', '12_main_0_1', 'main', '0', 12, 1)
('test', '13_main_1_3', 'main', '1', 7, 1)
('test', '13_main_2_1', 'main', '2', 8, 0)
('test', '17_assy_0_1', 'assy', '0', 9, 2)
('test', '17_assy_1_5', 'assy', '1', 8, 1)
('test', '18_assy_2_5', 'assy', '2', 7, 3)
('test', '19_assy_3_5', 'assy', '3', 8, 1)
('test', '23_assy_1_2', 'assy', '1', 8, 1)
('train', '01_assy_1_1', 'assy', '1', 8, 1)
('train', '02_assy_0_1', 'assy', '0', 11, 1)
('train', '02_assy_1_2', 'assy', '1', 8, 1)
('train', '04_assy_2_1', 'assy', '2', 8, 2)
('train', '06_a

Observation 4.2: Of the 53 recordings with instruction 0, only 5 have steps executed incorrectly: 02_assy_0_1, 08_assy_0_1, 08_main_0_1, 12_main_0_1 and 17_assy_0_1. Of the 31 recordings with other instructions, 25 have them. Instruction 0 is therefore the correct instruction, and the five exceptions are mistakes made while following it, as the paper reports.

The six recordings with another instruction and no incorrect step are 10_main_3_4, 13_main_2_1, 14_main_2_2, 14_main_2_3, 22_assy_2_3 and 25_assy_2_1. A missing step does not appear among the incorrect ones, so their errors may be omissions (hypothesis, to be checked on the final states).

The state of the components is annotated in PSR_labels_raw.csv, one row each time it changes. Two things are checked on it: how a removal appears, on the one recording where a component goes back from installed to not installed, and whether the recordings with an erroneous instruction but no incorrect step end in a different state from the correct ones.

In [5]:
# 4.3 Steps and state changes of one recording (05_assy_0_1)
def read_states(rec):
    """Frame and state vector (11 integers: 1 correct, 0 not installed, -1 incorrect) of every row of PSR_labels_raw.csv."""
    return [(r[0], [int(v) for v in r[1:]]) for r in read_rows(rec / "PSR_labels_raw.csv")]

def state_changes(previous, current):
    """Components whose value changes between two state vectors."""
    return [f"{COMPONENTS[i]} {a} -> {b}" for i, (a, b) in enumerate(zip(previous, current)) if a != b]

rec = RECORDINGS_DIR / "val" / "05_assy_0_1"
print("PSR_labels_with_errors.csv:")
for r in read_rows(rec / "PSR_labels_with_errors.csv"):
    print("  ", r)

states = read_states(rec)
print("PSR_labels_raw.csv, state changes:")
print("  ", states[0][0], "start", states[0][1])
for (_, previous), (frame, current) in zip(states, states[1:]):
    print("  ", frame, state_changes(previous, current))

PSR_labels_with_errors.csv:
   ('000521.jpg', '3', 'Install front chassis')
   ('000521.jpg', '6', 'Install front chassis pin')
   ('000783.jpg', '9', 'Install rear chassis')
   ('000783.jpg', '15', 'Install front rear chassis pin')
   ('000899.jpg', '18', 'Install rear rear chassis pin')
   ('001416.jpg', '30', 'Install rear wheel assy')
   ('002233.jpg', '17', 'Remove front rear chassis pin')
   ('002349.jpg', '15', 'Install front rear chassis pin')
   ('002456.jpg', '21', 'Install front bracket')
   ('002456.jpg', '24', 'Install front bracket screw')
   ('002870.jpg', '27', 'Install front wheel assy')
PSR_labels_raw.csv, state changes:
   000000.jpg start [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
   000521.jpg ['front chassis 0 -> 1', 'front chassis pin 0 -> 1']
   000783.jpg ['rear chassis 0 -> 1', 'front rear chassis pin 0 -> 1']
   000899.jpg ['rear rear chassis pin 0 -> 1']
   001416.jpg ['rear wheel assy 0 -> 1']
   002233.jpg ['front rear chassis pin 1 -> 0']
   002349.jpg ['front rea

Observation 4.3: In 05_assy_0_1 the front rear chassis pin is removed at frame 2233 and installed again at frame 2349, about 12 s later at 10 fps. Both steps are annotated in PSR_labels.csv as well (step 17, remove, and step 15, install), so a removal followed by a new installation is a correction during the execution, not an incorrect step. The order is flexible: the front bracket is installed after the rear wheel.

In [6]:
# 4.4 Final states: correct recordings against recordings with an erroneous instruction and no incorrect step
from collections import Counter

by_name = {r.name: r for r in recordings}
clean = [s for s in summary if s[3] == "0" and s[5] == 0]         # instruction 0 and no incorrect step

for task in ("assy", "main"):
    finals = Counter(tuple(read_states(by_name[s[1]])[-1][1]) for s in clean if s[2] == task)
    print(task, ":", sum(finals.values()), "correct recordings; most common final states:")
    for state, n in finals.most_common(3):
        print("  ", list(state), n)

print()
for name in ["10_main_3_4", "13_main_2_1", "14_main_2_2", "14_main_2_3", "22_assy_2_3", "25_assy_2_1"]:
    print(name, read_states(by_name[name])[-1][1])

assy : 24 correct recordings; most common final states:
   [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1] 24
main : 24 correct recordings; most common final states:
   [1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1] 24

10_main_3_4 [1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1]
13_main_2_1 [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1]
14_main_2_2 [1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1]
14_main_2_3 [1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1]
22_assy_2_3 [1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1]
25_assy_2_1 [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1]


Observation 4.4: The 48 recordings with the correct instruction and no incorrect step all end in the same state, 24 for each task. An assembly ends with every component installed except the short rear chassis. A maintenance ends with the rear chassis removed and the short rear chassis installed in its place.

Of the six recordings with an erroneous instruction and no incorrect step, two end without a pin (14_main_2_2, 25_assy_2_1), two maintenances end with the rear chassis still installed and no short rear chassis (13_main_2_1, and 14_main_2_3, which also lacks a pin), one assembly ends with the short rear chassis in place of the rear chassis (22_assy_2_3), and one ends in the correct final state (10_main_3_4).

The errors of IndustReal are therefore of several kinds: components installed incorrectly, components left out, the component of the other procedure installed in place of the right one, and errors that leave no trace in the final state. Why the two maintenances keep the rear chassis, whether the replacement was skipped, is not checked here: it would need their initial state (hypothesis).

The paper reports 35 videos with a missing or incorrect step, 38 incorrect step completions and 27 error states with a bounding box. They are checked here on the annotations. A recording has errors, as in the function video_contains_errors of the dataset code, if a step expected for its task is never completed, or if a component is ever installed incorrectly (a -1 in PSR_labels_raw.csv).

In [15]:
# 4.5 Recordings with errors, by the definition of the dataset authors
expected_steps = {task: {s["id"] for s in procedure if s[f"expected_in_{task}"]} for task in ("assy", "main")}

def find_errors(rec):
    """As video_contains_errors in PSR/psr_utils.py: expected steps never completed, and whether a component is ever incorrect."""
    task = rec.name.split("_")[1]
    completed = {int(r[1]) for r in read_rows(rec / "PSR_labels.csv")}
    missing = sorted(expected_steps[task] - completed)
    incorrect = any(-1 in state for _, state in read_states(rec))
    return missing, incorrect

recordings_with_errors = []
for rec in recordings:
    missing, incorrect = find_errors(rec)
    if missing or incorrect:
        recordings_with_errors.append((rec.name, missing, incorrect))

print(len(recordings_with_errors), "of", len(recordings), "recordings with errors (paper: 35)")
print(sum(s[5] for s in summary), "incorrect step completions (paper: 38)")
print()
for name, missing, incorrect in recordings_with_errors:
    print(name, "| missing steps:", missing, "| incorrect component:", incorrect)

35 of 84 recordings with errors (paper: 35)
38 incorrect step completions (paper: 38)

03_assy_1_3 | missing steps: [30] | incorrect component: True
08_assy_0_1 | missing steps: [24] | incorrect component: True
08_assy_2_4 | missing steps: [24] | incorrect component: True
08_main_0_1 | missing steps: [] | incorrect component: True
09_assy_3_1 | missing steps: [15] | incorrect component: True
10_assy_3_2 | missing steps: [18] | incorrect component: True
12_assy_3_4 | missing steps: [27] | incorrect component: True
12_main_0_1 | missing steps: [] | incorrect component: True
13_main_1_3 | missing steps: [30] | incorrect component: True
13_main_2_1 | missing steps: [12] | incorrect component: False
17_assy_0_1 | missing steps: [] | incorrect component: True
17_assy_1_5 | missing steps: [30] | incorrect component: True
18_assy_2_5 | missing steps: [21, 24] | incorrect component: True
19_assy_3_5 | missing steps: [30] | incorrect component: True
23_assy_1_2 | missing steps: [27] | incorrect 

Observation 4.5: With the definition of the dataset code, 35 of the 84 recordings have errors and the incorrect step completions are 38 in total, the two numbers of the paper. In 30 of them a component is installed incorrectly, and its correct installation then never follows in most of them. The other five have only missing steps: 13_main_2_1, 14_main_2_2, 14_main_2_3, 22_assy_2_3 and 25_assy_2_1. 10_main_3_4 has an erroneous instruction but no error in its annotations, so it counts as a recording without errors.

The steps most often missing are the installation of the rear wheel (30), of the front wheel (27), of the two pins of the rear chassis (15, 18) and of the front bracket screw (24).

------------------------------------------------------------------------------------------------------------------------------------------

References Section 4:

IndustReal code repository, PSR/procedure_info.json at commit ad86a35: the 33 steps and the fields install, state_idx, expected_in_assy and expected_in_main.

Schoonbeek et al. 2024, IndustReal (WACV), Section 4.4.2: the 11 components, in the order of the state vector, and the values 1, 0 and -1; Figure 1: parts, components and the maintenance task.

Schoonbeek et al. 2024, IndustReal (WACV), Section 1: procedural errors (e.g. omissions) and execution errors (e.g. wrong type of nut used); Section 4.3: participants also made mistakes with the correct instructions, and these were annotated.

Schoonbeek et al. 2024, IndustReal (WACV), Section 3.1: the procedural information P, a set of actions each described by information such as a description, an image of the completed step or relative positions between components; Section 4.4.3: 724 correct and 38 incorrect step completions, 35 videos with a missing or incorrect step.

IndustReal code repository, PSR/psr_utils.py at commit ad86a35: convert_states_to_steps (how changes of state become steps, including removals) and video_contains_errors (a video has errors if an expected step is missing or a component is ever installed incorrectly).

------------------------------------------------------------------------------------------------------------------------------------------

## Section 5 - RGB videos

In [7]:
# 5.1 Content of the video archive
import zipfile

VIDEOS_ZIP_PATH = DATA_DIR / "all_rgb_videos.zip"

with zipfile.ZipFile(VIDEOS_ZIP_PATH) as z:
    entries = z.infolist()
mp4 = [e for e in entries if e.filename.endswith(".mp4")]
print(len(entries), "entries,", len(mp4), "mp4 files,", round(sum(e.file_size for e in mp4) / 1e9, 2), "GB")
print("Other entries:", [e.filename for e in entries if not e.filename.endswith(".mp4")])
print("Folders of the mp4 files:", sorted({str(Path(e.filename).parent) for e in mp4}))

86 entries, 86 mp4 files, 5.05 GB
Other entries: []
Folders of the mp4 files: ['.']


In [8]:
# 5.2 Videos against recordings
video_names = {Path(e.filename).stem for e in mp4}
recording_names = {r.name for r in recordings}
print("Videos without annotations:", sorted(video_names - recording_names))
print("Recordings without a video:", sorted(recording_names - video_names))

Videos without annotations: ['15_main_0_1', '21_assy_1_1']
Recordings without a video: []


Observation 5.2: The archive holds 86 videos, one for each of the 84 annotated recordings and two more: 15_main_0_1 and 21_assy_1_1. These are the two recordings missing from the nine zip files, so participant 15 did record a correct maintenance and participant 21 an instruction with errors, as the paper describes, but neither has frames or annotations. Neither the readme nor the paper says why. They are left out of every selection, since nothing can be measured on them.

In [11]:
# 5.3 Download, integrity check and extraction of the videos (only what is missing)
import hashlib
import shutil
import urllib.request

def md5_of(path, block_size=2**24):
    """MD5 of a file, read in blocks of 16 MB."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        while block := f.read(block_size):
            h.update(block)
    return h.hexdigest()

if VIDEOS_DIR.is_dir() and len(list(VIDEOS_DIR.glob("*.mp4"))) == len(mp4):
    print("All videos already extracted, no download")
else:
    if not VIDEOS_ZIP_PATH.exists():
        print("Downloading the video archive (about 5 GB)")
        with urllib.request.urlopen(DATASET_URL + VIDEOS_ZIP_ID) as r, open(VIDEOS_ZIP_PATH, "wb") as f:
            shutil.copyfileobj(r, f, 2**24)
    digest = md5_of(VIDEOS_ZIP_PATH)
    assert digest == VIDEOS_ZIP_MD5, f"MD5 mismatch: {digest}"
    print("MD5 matches the dataset page")
    with zipfile.ZipFile(VIDEOS_ZIP_PATH) as z:
        z.extractall(VIDEOS_DIR)
print("Videos in", VIDEOS_DIR, ":", len(list(VIDEOS_DIR.glob("*.mp4"))))

MD5 matches the dataset page
Videos in c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\industreal\videos : 86


Observation 5.3: The 86 videos are extracted into data/industreal/videos, after the MD5 of the archive matched the one on the dataset page. A second run does nothing.

The readme gives 1080x720 for the RGB frames, the paper 1280x720 (Section 4.2). The size of one frame of the dataset is read here, together with the size, frame rate and length of the video of the same recording.

In [16]:
# 5.4 Resolution and frame rate: one RGB frame of the dataset against its video
import io
import cv2
from PIL import Image

rec_name = "05_assy_0_1"
with RemoteZip(DATASET_URL + ZIP_IDS["val_p1"]) as z:            # reads only the index and one frame
    rgb_names = sorted(n for n in z.namelist() if n.startswith(rec_name + "/rgb/") and n.endswith(".jpg"))
    frame = Image.open(io.BytesIO(z.read(rgb_names[0])))
print(rec_name, ":", len(rgb_names), "RGB frames, first frame", frame.size, "(width, height)")

video = cv2.VideoCapture(str(VIDEOS_DIR / f"{rec_name}.mp4"))
print("Video:", int(video.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(video.get(cv2.CAP_PROP_FRAME_HEIGHT)),
      "|", video.get(cv2.CAP_PROP_FPS), "fps |", int(video.get(cv2.CAP_PROP_FRAME_COUNT)), "frames")
video.release()

05_assy_0_1 : 2918 RGB frames, first frame (1280, 720) (width, height)
Video: 1280 x 720 | 10.0 fps | 2918 frames


Observation 5.4: The RGB frames of 05_assy_0_1 are 1280x720, as in the paper; the 1080x720 of the readme is an error. Its video has the same size, runs at 10 fps and has 2918 frames, as many as the jpg frames, so the frame n of the video corresponds to the jpg frame n. The video is compressed, so its pixels are not identical to those of the jpg: the videos serve for looking at the recordings, the jpg frames for annotation and measurement.

In [19]:
# 5.5 All videos against the RGB frames of their recordings
if RGB_COUNTS_PATH.exists():
    rgb_counts = json.loads(RGB_COUNTS_PATH.read_text())
else:
    rgb_counts = {}
    for part, file_id in ZIP_IDS.items():
        with RemoteZip(DATASET_URL + file_id) as z:               # reads only the index of each zip
            for n in z.namelist():
                parts = n.split("/")
                if len(parts) == 3 and parts[1] == "rgb" and n.endswith(".jpg"):
                    rgb_counts[parts[0]] = rgb_counts.get(parts[0], 0) + 1
    RGB_COUNTS_PATH.write_text(json.dumps(rgb_counts, indent=1))

differences = []
for rec_name, n_frames in sorted(rgb_counts.items()):
    video = cv2.VideoCapture(str(VIDEOS_DIR / f"{rec_name}.mp4"))
    info = (int(video.get(cv2.CAP_PROP_FRAME_WIDTH)), int(video.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            video.get(cv2.CAP_PROP_FPS), int(video.get(cv2.CAP_PROP_FRAME_COUNT)))
    video.release()
    if info != (1280, 720, 10.0, n_frames):
        differences.append((rec_name, info, n_frames))

print(len(rgb_counts), "recordings,", sum(rgb_counts.values()), "RGB frames in total")
print("Videos that differ from their frames (size, fps or length):", differences)

84 recordings, 207072 RGB frames in total
Videos that differ from their frames (size, fps or length): [('11_assy_0_1', (1280, 720, 10.0, 2572), 2571)]


In [21]:
# 5.6 Frames of 11_assy_0_1 counted by decoding the video
video = cv2.VideoCapture(str(VIDEOS_DIR / "11_assy_0_1.mp4"))
decoded = 0
while video.grab():                                                # grab() decodes the next frame without converting it
    decoded += 1
video.release()
print("Decoded frames:", decoded, "| jpg frames:", rgb_counts["11_assy_0_1"])

Decoded frames: 2572 | jpg frames: 2571


Observation 5.5-5.6: The 84 recordings hold 207,072 RGB frames, so the 26.9K frames annotated for ASD are 13% of them, as in the paper (Section 4.4.2). 83 videos are 1280x720 at 10 fps with as many frames as their jpg frames. The video of 11_assy_0_1 has one frame more, 2572 against 2571, also when its frames are counted by decoding, so its frame count is not an error of the file header. Where the extra frame lies is not checked.

The videos are therefore used only for looking. A frame chosen on a video is always confirmed on the jpg frame of the same index before it is annotated, so that an offset like the one of 11_assy_0_1 cannot enter the annotations.

-------------------------------------

References Section 5:

Schoonbeek et al. 2024, IndustReal (WACV), Section 4.2: RGB camera at 1280x720, 10 fps; Section 4.4.2: 26.9K frames (13% of the total) annotated for ASD.

IndustReal code repository, readme (Data information): RGB at 1080x720 and 10 fps, the size contradicted by the frames. IndustReal on 4TU.ResearchData (doi 10.4121/b008dd74-020d-4ea4-a8ba-7bb60769d224): all_rgb_videos.zip and its MD5.

IndustReal issue #7 (github.com/TimSchoonbeek/IndustReal/issues/7): the dataset page was unreachable for some days in November 2026, a reason to keep a local copy of what is used.

-------------------------------------

## Section 6 - Action annotations

AR_labels.csv lists the actions of each recording: an action id, its name, made of a verb and a noun, and the first and last frame. The paper reports 75 action classes, 9,273 instances, an average duration of 1.9 ± 1.4 s, 110 ± 38 actions per recording (134 ± 32 in assembly, 79 ± 13 in maintenance) and 24.2% of the instances overlapping another one (Section 4.4.1); the supplementary material gives 73 classes in its text and 75 in its Figure 4. These numbers are checked on the annotations.

In [23]:
# 6.1 Action annotations: classes, instances, durations and overlaps
import statistics

actions = []                                                       # (recording, task, action id, name, first frame, last frame)
for rec in recordings:
    for r in read_rows(rec / "AR_labels.csv"):
        actions.append((rec.name, rec.name.split("_")[1], int(r[1]), r[2], int(r[3][:-4]), int(r[4][:-4])))

names_by_id = {}
for a in actions:
    names_by_id.setdefault(a[2], set()).add(a[3])
print(len(actions), "action instances (paper: 9,273)")
print(len(names_by_id), "action ids, from", min(names_by_id), "to", max(names_by_id), "(paper: 75 classes)")
print("Ids with more than one name:", {i: n for i, n in names_by_id.items() if len(n) > 1})
print("Verbs:", sorted({n.split("_")[0] for names in names_by_id.values() for n in names}))

durations = [(a[5] - a[4]) / FPS for a in actions]
print(f"Duration: {statistics.mean(durations):.1f} ± {statistics.stdev(durations):.1f} s (paper: 1.9 ± 1.4 s)")

for label, task in [("all", None), ("assembly", "assy"), ("maintenance", "main")]:
    counts = [sum(a[0] == rec.name for a in actions) for rec in recordings if task in (None, rec.name.split("_")[1])]
    print(f"Actions per recording, {label}: {statistics.mean(counts):.0f} ± {statistics.stdev(counts):.0f}")

overlapping = 0
for rec in recordings:
    own = [a for a in actions if a[0] == rec.name]
    overlapping += sum(any(b is not a and a[4] < b[5] and b[4] < a[5] for b in own) for a in own)
print(f"Instances overlapping another one: {100 * overlapping / len(actions):.1f}% (paper: 24.2%)")

9273 action instances (paper: 9,273)
74 action ids, from 0 to 74 (paper: 75 classes)
Ids with more than one name: {}
Verbs: ['align', 'browse', 'check', 'fit', 'loosen', 'plug', 'pull', 'put', 'take', 'tighten']
Duration: 1.9 ± 1.5 s (paper: 1.9 ± 1.4 s)
Actions per recording, all: 110 ± 38
Actions per recording, assembly: 134 ± 33
Actions per recording, maintenance: 78 ± 14
Instances overlapping another one: 42.7% (paper: 24.2%)


In [24]:
# 6.2 Missing action id, class list, and the overlap measured in other ways
from collections import Counter

all_ids = set(range(min(names_by_id), max(names_by_id) + 1))
print("Ids never used:", sorted(all_ids - set(names_by_id)))
counts = Counter(a[2] for a in actions)
for i in sorted(names_by_id):
    print(i, next(iter(names_by_id[i])), counts[i])

pairs, long_overlap, nested = 0, set(), set()
for rec in recordings:
    own = [a for a in actions if a[0] == rec.name]
    for i, a in enumerate(own):
        for b in own[i + 1:]:
            shared = min(a[5], b[5]) - max(a[4], b[4])              # frames shared by the two intervals
            if shared > 0:
                pairs += 1
                if shared >= FPS:
                    long_overlap |= {id(a), id(b)}
                if (a[4] <= b[4] and b[5] <= a[5]) or (b[4] <= a[4] and a[5] <= b[5]):
                    nested |= {id(a), id(b)}
print()
print(f"Overlapping pairs over all instances: {100 * pairs / len(actions):.1f}%")
print(f"Instances overlapping another one by at least 1 s: {100 * len(long_overlap) / len(actions):.1f}%")
print(f"Instances in a nested pair (one inside the other): {100 * len(nested) / len(actions):.1f}%")

Ids never used: [37]
0 take_short_brace 134
1 align_objects 569
2 take_pin_short 245
3 plug_short_pin 251
4 take_tooth_washer 395
5 take_nut 437
6 tighten_nut 464
7 check_instruction 1300
8 take_partial_model 302
9 take_long_brace 119
10 take_screw_pin 73
11 take_instruction 3
12 put_instruction 4
13 take_pin_long 59
14 put_pin_long 15
15 take_wing_beam 61
16 plug_screw_pin 61
17 take_round_washer 358
18 take_acorn_nut 58
19 tighten_acorn_nut 61
20 take_pin_middle 76
21 take_wheel 298
22 plug_pin_long 46
23 take_wing 74
24 put_wing 17
25 plug_partial_model 4
26 plug_pin_middle 81
27 take_pulley 107
28 plug_wheel 8
29 browse_instruction 134
30 fit_short_brace 110
31 fit_tooth_washer 386
32 fit_round_washer 292
33 fit_long_brace 18
34 fit_nut 414
35 put_screw_pin 20
36 put_wheel 76
38 pull_wheel 9
39 loosen_nut 128
40 put_nut 90
41 pull_objects 53
42 put_pin_middle 28
43 take_objects 321
44 put_partial_model 296
45 put_objects 201
46 pull_pin_short 90
47 put_pin_short 77
48 put_long_brac

Observation 6.1-6.2: The annotations hold 9,273 action instances, as in the paper. Their duration is 1.9 ± 1.5 s and their number per recording 110 ± 38, 134 ± 33 in assembly and 78 ± 14 in maintenance, equal to the paper up to rounding. Every id has a single name.

The ids go from 0 to 74, but id 37 is never used: 75 classes are defined and 74 occur. The 75 of the paper matches the defined classes; the 73 of the text of the supplementary material is not found in the data. Only 10 of the 12 verbs of MECCANO occur: screw and unscrew are never used, and the screw pin is installed with plug. The most frequent classes are check_instruction (1,300) and align_objects (569), and 23 classes cover 80% of the instances, as in the supplementary material; three classes occur only once.

The share of instances overlapping another one is 42.7% when any shared frame counts, against 24.2% in the paper. Neither the overlapping pairs over all instances (26.7%), nor an overlap of at least 1 s (14.8%), nor nested pairs (10.3%) gives the number of the paper, whose definition is not stated. The difference is left unexplained, since the actions are not an input of the pipeline of this thesis.

-------------------------------------

References Section 6:

Schoonbeek et al. 2024, IndustReal (WACV), Section 4.4.1: verbs adopted from MECCANO, 75 action classes, 9,273 instances, duration 1.9 ± 1.4 s, 110 ± 38 actions per recording (134 ± 32 assembly, 79 ± 13 maintenance), 24.2% overlapping instances.

Schoonbeek et al. 2024, supplementary material (arXiv 2310.17323), Section 7.2.1: start frames at the initiation of the action, annotation with ELAN by two annotators who reviewed each other; 73 classes in the text and 75 in Figure 4; check_instruction and align_objects the most frequent classes, 23 classes covering 80% of the data.

-------------------------------------